In [2]:
import pandas as pd
import networkx as nx

cast = pd.read_csv(
    "data/processed/production_cast.csv"
)

cast["performer_id"] = (
    "A_" + cast["performer_id"].astype(str)
)

cast_clean = cast[
    cast["performer_id"] != "A_60670"
].copy()

cast_clean.to_csv(
    "data/processed/production_cast_clean.csv",
    index=False
)

cast_clean.shape

(115633, 4)

In [3]:
keywords = [
    "spelvin",
    "anonymous",
    "unknown",
    "uncredited",
    "ensemble"
]

mask = cast_clean["performer_name"].str.contains(
    "|".join(keywords),
    case=False,
    na=False
)

cast_clean.loc[mask, [
    "performer_id",
    "performer_name"
]].drop_duplicates()

,performer_id,performer_name
6958,A_105848,"George Spelvin, Jr."
7610,A_60669,Frank Spelvin
11675,A_60671,Georgette Spelvin
14931,A_446026,Joseph Spelvin
18689,A_60672,James Spelvin
21472,A_39646,Rooney Ensemble
21483,A_39647,Thermein Ensemble
28727,A_382817,George Spelvinsky
37246,A_39645,Iza Volpin's Ensemble
49865,A_105617,Dummy Spelvin


In [4]:
name_variation = (
    cast_clean
    .groupby("performer_id")["performer_name"]
    .nunique()
    .sort_values(ascending=False)
)

name_variation.head(20)

performer_id
A_100000    1
A_58063     1
A_58070     1
A_58074     1
A_58077     1
A_58079     1
A_58087     1
A_58088     1
A_58089     1
A_58090     1
A_58091     1
A_58092     1
A_58094     1
A_58096     1
A_58097     1
A_58100     1
A_58101     1
A_58105     1
A_58107     1
A_58111     1
Name: performer_name, dtype: int64

In [5]:
cast_clean[
    cast_clean["performer_id"].isin(
        name_variation.head(20).index
    )
].sort_values("performer_id")

,performer_id,performer_name,character,production_id
54130,A_100000,Gwyneth Hughes,Sarah Pugh,1289
66086,A_100000,Gwyneth Hughes,Sarah Pugh,476323
64614,A_100000,Gwyneth Hughes,Mrs. Resurrection Jones,2033
49655,A_100000,Gwyneth Hughes,Sarah Pugh,1069
32933,A_58063,Florence Robinson,Gladys Kelcey,11923
26587,A_58063,Florence Robinson,Attendant,11677
27560,A_58063,Florence Robinson,Passer-by,13291
368,A_58070,Grace Robinson,Baby Dollin 'The Fairy Doll',10348
23604,A_58074,John Robinson,Boy Neighbor,11477
441,A_58077,Kathleen Robinson,Agatha Sterling,10353


In [6]:
spelvin_mask = cast_clean["performer_name"].str.contains(
    "spelvin",
    case=False,
    na=False
)

spelvin_df = cast_clean[spelvin_mask]

(
    spelvin_df
    .groupby(
        ["performer_id", "performer_name"]
    )
    .size()
    .sort_values(ascending=False)
)

performer_id  performer_name     
A_60671       Georgette Spelvin      5
A_105848      George Spelvin, Jr.    3
A_105617      Dummy Spelvin          1
A_107431      Giorgio Spelvino       1
A_111173      Abraham J. Spelvin     1
A_382817      George Spelvinsky      1
A_446026      Joseph Spelvin         1
A_468948      Fred Spelvin           1
A_471354      Milton Spelvin         1
A_484299      Harold Spelvin         1
A_60669       Frank Spelvin          1
A_60672       James Spelvin          1
dtype: int64

In [7]:
# Final cleaning checkpoint

# Remove confirmed placeholder identities
removed_performers = [
    "A_60670",   # George Spelvin
    "A_105617"   # Dummy Spelvin
]

cast_clean_final = cast[
    ~cast["performer_id"].isin(removed_performers)
].copy()

# Save cleaned cast dataset
cast_clean_final.to_csv(
    "data/processed/production_cast_clean.csv",
    index=False
)

# Summary
print("Cleaning complete!")
print()
print(f"Original cast rows: {len(cast):,}")
print(f"Clean cast rows: {len(cast_clean_final):,}")
print(f"Rows removed: {len(cast) - len(cast_clean_final):,}")
print()
print(f"Original performers: {cast['performer_id'].nunique():,}")
print(f"Clean performers: {cast_clean_final['performer_id'].nunique():,}")
print(f"Performers removed: {cast['performer_id'].nunique() - cast_clean_final['performer_id'].nunique():,}")

Cleaning complete!

Original cast rows: 115,677
Clean cast rows: 115,632
Rows removed: 45

Original performers: 55,263
Clean performers: 55,261
Performers removed: 2


# Cleaning decisions:
- Removed George Spelvin (A_60670) because it represents a placeholder identity rather than an individual performer and artificially inflated network centrality.
- Removed Dummy Spelvin (A_105617) as an explicit placeholder-style identity.
- Retained other Spelvin-named performers because no evidence indicated they were artefacts.
- Tested duplicate performer names across multiple IDs and found no overlapping production credits, suggesting separate individuals rather than duplicate identities.